In [0]:
# File: src/myfunctions.py
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, regexp_replace, coalesce, lit
from pyspark.sql.types import *

# Create SparkSession for files outside notebooks
spark = SparkSession.builder \
    .appName('data-transformations') \
    .getOrCreate()

# Utility functions for data validation
def tableExists(tableName, dbName):
    """Check if a table exists in the specified database"""
    try:
        return spark.catalog.tableExists(f"{dbName}.{tableName}")
    except Exception:
        return False

def columnExists(dataFrame, columnName):
    """Check if a column exists in the given DataFrame"""
    return columnName in dataFrame.columns

def numRowsInColumnForValue(dataFrame, columnName, columnValue):
    """Count rows for a specific value in a column"""
    df = dataFrame.filter(col(columnName) == columnValue)
    return df.count()

# Data transformation functions
def clean_customer_data(df):
    """
    Clean customer data by:
    - Standardizing name format (uppercase)
    - Removing spaces from email
    - Formatting phone numbers (digits only)
    - Handling null status values
    """
    return df.select(
        col("customer_id"),
        upper(col("first_name")).alias("first_name"),
        upper(col("last_name")).alias("last_name"),
        regexp_replace(col("email"), r'\s+', '').alias("email"),
        regexp_replace(col("phone"), r'[^\d]', '').alias("phone"),
        coalesce(col("status"), lit("ACTIVE")).alias("status")
    )

def calculate_customer_metrics(df):
    """Calculate customer metrics from transaction data"""
    from pyspark.sql.functions import sum as spark_sum, avg as spark_avg, count as spark_count
    
    return df.groupBy("customer_id") \
        .agg(
            spark_sum("transaction_amount").alias("total_spent"),
            spark_avg("transaction_amount").alias("avg_transaction"),
            spark_count("transaction_id").alias("transaction_count")
        )

# Data quality validation functions
def validate_data_quality(df, required_columns):
    """Validate that DataFrame has required columns and no null IDs"""
    issues = []
    
    # Check required columns exist
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        issues.append(f"Missing columns: {missing_columns}")
    
    # Check for null customer_ids
    if "customer_id" in df.columns:
        null_count = df.filter(col("customer_id").isNull()).count()
        if null_count > 0:
            issues.append(f"Found {null_count} null customer_ids")
    
    return {
        "is_valid": len(issues) == 0,
        "issues": issues,
        "total_rows": df.count()
    }